# Vietnamese Budget Forcing — Kaggle Runner

**Research question:** Does Test-Time Scaling via Budget Forcing transfer to Vietnamese language reasoning?

**Before running:**
1. Settings → Accelerator → **GPU T4 x1** (or P100)
2. Settings → Internet → **On**
3. Add your HuggingFace token: Settings → Secrets → Add `HF_TOKEN`

**Run order:** Cell 1 (GPU check) → Cell 2 (clone) → Cell 3 (install) → Cell 4 (HF auth) → Cell 5 (validate benchmarks) → Cell 6 (VRAM check) → Cell 7 (run sweep) → Cell 8 (aggregate) → Cell 9 (copy outputs)

**Scope:** BF-only, n_wait ∈ {0,1,2}. No retrieval.

In [1]:
# Cell 1 — Check GPU and environment
import subprocess, os, sys

print('=== GPU ===')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], check=False)

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

WORK_DIR = '/kaggle/working'
print(f'\nWorking dir: {WORK_DIR}')
print('Disk free: ', end='')
subprocess.run(['df', '-h', WORK_DIR])

=== GPU ===
name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14913 MiB
Tesla T4, 15360 MiB, 14913 MiB



PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB

Working dir: /kaggle/working
Disk free: Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  128K   20G   1% /kaggle/working


CompletedProcess(args=['df', '-h', '/kaggle/working'], returncode=0)

In [2]:
import os
import shutil

REPO_DIR = '/kaggle/working/s1_budget_forcing'
INPUT_PATH = '/kaggle/input/s1-budget-forcing'

# 1. If it's already there (e.g., interactive re-runs), skip setup
if os.path.exists(REPO_DIR):
    print(f'Repo already exists at {REPO_DIR}')

# 2. Try Option B first: If a Kaggle Dataset is attached, copy it locally
elif os.path.exists(INPUT_PATH):
    shutil.copytree(INPUT_PATH, REPO_DIR)
    print(f'Copied from Kaggle dataset to {REPO_DIR}')

# 3. Try Option A: Fallback to GitHub cloning if no dataset is attached
else:
    print(f'Dataset not found. Cloning from GitHub to {REPO_DIR}...')
    # Using subprocess to handle cloning reliably in background runs
    import subprocess
    result = subprocess.run(['git', 'clone', 'https://github.com/ajvan2808/s1_budget_forcing.git', REPO_DIR], capture_output=True, text=True)
    
    if result.returncode == 0:
        print("GitHub clone successful!")
    else:
        print(result.stderr)
        raise FileNotFoundError('Failed to clone from GitHub and no Kaggle Dataset was found.')

# Always move to the directory at the end
os.chdir(REPO_DIR)
print(f'CWD set to: {os.getcwd()}')

Dataset not found. Cloning from GitHub to /kaggle/working/s1_budget_forcing...


GitHub clone successful!
CWD set to: /kaggle/working/s1_budget_forcing


In [3]:
# Cell 3 — Install dependencies
# Kaggle has torch/transformers pre-installed; only install what's missing.
# No RAG dependencies needed — BF-only study.

!git checkout origin/feature

!uv sync

Note: switching to 'origin/feature'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 77a7b0b fix running devices


Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv


Resolved 180 packages in 25ms


⠙ Preparing packages... (0/168)                                                 

⠙ Preparing packages... (0/168)
⠙ Preparing packages... (0/168)
⠙ Preparing packages... (0/168)
pyzmq                ------------------------------     0 B/834.13 KiB
⠙ Preparing packages... (0/168)
pyzmq                ------------------------------     0 B/834.13 KiB
⠙ Preparing packages... (0/168)
pyzmq                ------------------------------     0 B/834.13 KiB
⠙ Preparing packages... (0/168)
ipython              ------------------------------     0 B/812.35 KiB
pyzmq                ------------------------------     0 B/834.13 KiB
⠙ Preparing packages... (0/168)
ipython              ------------------------------     0 B/812.35 KiB
pyzmq                ------------------------------     0 B/834.13 KiB
⠙ Preparing packages... (0/168)
ipython              ------------------------------     0 B/812.35 KiB
pyzmq                ------------------------------     0 B/834.13 KiB
⠙ Preparing packages... (0/168)
ipython              ------------------------------     0 B/812.35 KiB
py

...

███████████████████░ [165/168] nvidia-cublas==13.1.1.3                          

███████████████████░ [166/168] nvidia-nvshmem-cu13==3.4.5                       

███████████████████░ [167/168] bitsandbytes==0.49.2                             

████████████████████ [168/168] torch==2.12.0                                    

Installed 168 packages in 9.51s
 + accelerate==1.13.0
 + aiohappyeyeballs==2.6.2
 + aiohttp==3.14.0
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + anyio==4.13.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + async-lru==2.3.0
 + async-timeout==5.0.1
 + attrs==26.1.0
 + babel==2.18.0
 + beautifulsoup4==4.14.3
 + bitsandbytes==0.49.2
 + bleach==6.3.0
 + certifi==2026.5.20
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.4.1
 + comm==0.2.3
 + contourpy==1.3.2
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.5.5
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + datasets==4.8.5
 + debugpy==1.8.21
 + decorator==5.3.1
 + defusedxml==0.7.1
 + dill==0.4.1
 + exceptiongroup==1.3.1
 + executing==2.2.1
 + fastjsonschema==2.21.2
 + filelock==3.29.1
 + fonttools==4.63.0
 + fqdn==1.5.1
 + frozenlist==1.8.0
 + fsspec==2026.2.0
 + h11==0.16.0
 + hf-xet==1.5.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.17.0
 + idna==3.18
 + ipykernel==7.2.0
 +

In [4]:
# Cell 4 — HuggingFace authentication
# Required for r1-distill-7B and Vietnamese models.
# Store your token in Kaggle Secrets as HF_TOKEN — never hardcode it here.
from kaggle_secrets import UserSecretsClient
import os

try:
    secret = UserSecretsClient()
    hf_token = secret.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace login successful.')
except Exception as e:
    print(f'[WARN] HF auth skipped: {e}')
    print('Public models (Qwen2.5, SeaLLM) will still work without a token.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace login successful.


In [5]:
!uv pip install bitsandbytes>=0.46.1  

Using Python 3.12.13 environment at: /usr


⠙ Resolving dependencies...                                                     

⠹ bitsandbytes==0.49.2                                                          

⠹ torch==2.10.0+cu128                                                           

⠸ torch==2.10.0+cu128                                                           

...

⠙ Preparing packages... (0/1)
cuda-bindings        ------------------------------ 4.70 MiB/11.59 MiB          

⠙ Preparing packages... (0/1)
cuda-bindings        ------------------------------ 6.76 MiB/11.59 MiB          

⠹ Preparing packages... (0/1)
cuda-bindings        ------------------------------ 9.53 MiB/11.59 MiB          

⠹ Preparing packages... (0/1)
Prepared 1 package in 316ms
Uninstalled 1 package in 12ms
░░░░░░░░░░░░░░░░░░░░ [0/2] Installing wheels...                                 

Installed 2 packages in 280ms
 + bitsandbytes==0.49.2
 - cuda-bindings==13.2.0
 + cuda-bindings==12.9.4


In [6]:
# Cell 5 — Validate Vietnamese benchmarks
import sys, os
sys.path.insert(0, '/kaggle/working/s1_budget_forcing/experiments')
os.chdir('/kaggle/working/s1_budget_forcing')

!python experiments/data/download_vi_benchmarks.py --n_samples 2


Benchmark: vi_gsm8k
  Multilingual GSM8K — Vietnamese split (250 problems)
  Source: hllj/vi_gsm8k


README.md: 1.85kB [00:00, 3.69MB/s]


vi_train.json: 0.00B [00:00, ?B/s]

vi_train.json: 5.50MB [00:00, 74.8MB/s]


vi_test.json: 1.01MB [00:00, 125MB/s]
Generating train split:   0%|                   | 0/7473 [00:00<?, ? examples/s]

...

train.jsonl: 106kB [00:00, 31.0MB/s]


valid.jsonl: 273kB [00:00, 109MB/s]


test.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 3.74MB [00:00, 142MB/s]
Generating test split:   0%|                    | 0/9833 [00:00<?, ? examples/s]

Generating test split: 100%|█████| 9833/9833 [00:00<00:00, 306515.58 examples/s]
[OK]  Loaded. Size: 9833 samples
      Fields: ['id', 'question', 'choices', 'answer']
[OK]  All expected fields present: ['question', 'answer', 'choices', 'id']

--- First 2 samples ---

Sample 1:
  id: '28-0021'
  question: 'Một nền kinh tế trong trạng thái toàn dụng nhân công có nghĩa là:'
  choices: ['A. Không còn lạm phát nhưng có thể còn thất nghiệp', 'B. Không còn thất nghiệp nhưng có thể còn lạm phát', 'C. Không còn thất nghiệp và không còn lạm phát', 'D. Vẫn còn một tỷ lệ lạm phát và tỷ lệ thất nghiệp nhất định']
  answer: None

Sample 2:
  id: '28-0022'
  question: 'Trong cơ chế tỷ giá hối đoái cố định, muốn triệt tiêu lượng dư cung ngoại tệ, NHTƯ  phải:'
  choices: ['A. Dùng ngoại tệ để mua nội tệ', 'B. Dùng nội tệ để mua ngoại tệ', 'C. Không can thiệp vào thị trường ngoại hối', 'D. Các lựa chọn đều sai']
  answer: None

Summary:
  [OK]   vi_gsm8k
  [OK]   vimmlu

All benchmarks validated succes

In [7]:
# Cell 6 — VRAM check
# Verify each model fits in T4 VRAM (15 GB) under 4-bit quantization.
# r1-distill-7B and Vi-7B models: ~3.5 GB each at 4-bit.
# Run this before the sweep to catch OOM before wasting GPU time.
import sys, os
sys.path.insert(0, '/kaggle/working/s1_budget_forcing/experiments')
os.chdir('/kaggle/working/s1_budget_forcing')

from models.model_loader import SUPPORTED_MODELS, estimate_vram_gb
import torch

VI_MODELS = ['qwen2.5-3B', 'r1-distill-7B', 'vinallama-7b', 'vistral-7b', 'seallm-7b']

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU VRAM: {total_vram:.1f} GB')
else:
    total_vram = 0
    print('[WARN] No GPU detected — models will run on CPU (very slow)')

print()
print(f'{"Model":<20} {"HF ID":<45} {"VRAM 4-bit":>10} {"Fits T4":>8}')
print('-' * 90)
for name in VI_MODELS:
    hf_id = SUPPORTED_MODELS.get(name, 'NOT IN REGISTRY')
    vram = estimate_vram_gb(name, load_in_4bit=True)
    fits = '✓' if vram < total_vram * 0.9 else '✗ OOM risk'
    print(f'{name:<20} {hf_id:<45} {vram:>8.1f} GB {fits:>8}')

print()
print('Note: Vinallama/Vistral use </s> as EoT token (LLaMA-2/Mistral base).')
print('      If thinking_tokens stays 0 after n_wait>0, check EoT suppression in decoding.py.')

GPU VRAM: 15.6 GB

Model                HF ID                                         VRAM 4-bit  Fits T4
------------------------------------------------------------------------------------------
qwen2.5-3B           Qwen/Qwen2.5-3B-Instruct                           1.5 GB        ✓
r1-distill-7B        deepseek-ai/DeepSeek-R1-Distill-Qwen-7B            3.5 GB        ✓
vinallama-7b         vilm/vinallama-7b-chat                             3.5 GB        ✓
vistral-7b           vilm/vistral-7b-chat                               3.5 GB        ✓
seallm-7b            SeaLLMs/SeaLLMs-v3-7B-Chat                         3.5 GB        ✓

Note: Vinallama/Vistral use </s> as EoT token (LLaMA-2/Mistral base).
      If thinking_tokens stays 0 after n_wait>0, check EoT suppression in decoding.py.


In [8]:
# Cell 7 — Run experiments
#
# Set PRESET below, then Run All from this cell.
# Start with SMOKE to confirm pipeline, then scale up.
#
# SMOKE        1 model × 1 benchmark × 3 n_wait × 5 samples  → ~5 min
# SMALL_MATRIX 2 models × 2 benchmarks × 3 n_wait × 20 samples → ~30-45 min
# FULL         5 models × 2 benchmarks × 3 n_wait × 100 samples → ~3-5 hr

import os, subprocess
os.chdir('/kaggle/working/s1_budget_forcing')

# ── Choose a preset ──────────────────────────────────────────────────────────
PRESET = 'SMALL_MATRIX'   # 'SMOKE' | 'SMALL_MATRIX' | 'FULL'

PRESETS = {
    'SMOKE': dict(
        MODELS      = 'qwen2.5-3B',
        BENCHMARKS  = 'vi_gsm8k',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '5',
        EXTRA_ARGS  = '--max_tokens 512',   # 4-bit ON by default on Kaggle GPU
    ),
    'SMALL_MATRIX': dict(
        MODELS      = 'qwen2.5-3B r1-distill-7B',
        BENCHMARKS  = 'vi_gsm8k vimmlu',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '20',
        EXTRA_ARGS  = '--max_tokens 512',   # cap per-sample time; keeps run within 12h
    ),
    'FULL': dict(
        MODELS      = 'qwen2.5-3B r1-distill-7B vinallama-7b vistral-7b seallm-7b',
        BENCHMARKS  = 'vi_gsm8k vimmlu',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '100',
        EXTRA_ARGS  = '',
    ),
}

cfg = PRESETS[PRESET]
print(f'Running preset: {PRESET}')
for k, v in cfg.items():
    print(f'  {k}={v!r}')

env = os.environ.copy()
env.update({
    'MODELS':      cfg['MODELS'],
    'BENCHMARKS':  cfg['BENCHMARKS'],
    'N_WAIT_LIST': cfg['N_WAIT_LIST'],
    'N_SAMPLES':   cfg['N_SAMPLES'],
    'OUTPUT_DIR':  'experiments/results',
    'EXTRA_ARGS':  cfg['EXTRA_ARGS'],
})

result = subprocess.run(
    ['bash', 'experiments/scripts/run_vi_bf.sh'],
    env=env,
    cwd='/kaggle/working/s1_budget_forcing',
)
print(f'\nExit code: {result.returncode}')

Running preset: SMALL_MATRIX
  MODELS='qwen2.5-3B r1-distill-7B'
  BENCHMARKS='vi_gsm8k vimmlu'
  N_WAIT_LIST='0 1 2'
  N_SAMPLES='20'
  EXTRA_ARGS=''
Vietnamese Budget Forcing Sweep (BF-only)
  Models:     qwen2.5-3B r1-distill-7B
  Benchmarks: vi_gsm8k vimmlu
  n_wait:     0 1 2
  n_samples:  20
  output_dir: experiments/results
  seed:       42

----------------------------------------------------------------------
  Model: qwen2.5-3B  |  Benchmark: vi_gsm8k
----------------------------------------------------------------------
  Running python experiments/evaluation/run_eval_vi.py --model qwen2.5-3B --benchmark vi_gsm8k ...



[run_eval_vi] Started at 20260613_170306
  model=qwen2.5-3B, benchmark=vi_gsm8k
  n_wait_list=[0, 1, 2], n_samples=20
  trigger='Chờ một chút', load_in_4bit=True, max_new_tokens=2048
[run_eval_vi] Loading benchmark: hllj/vi_gsm8k
[run_eval_vi] Sampled 20 questions (seed=42)
[run_eval_vi] Loading model: qwen2.5-3B
Loading: Qwen/Qwen2.5-3B-Instruct | 4-bit=True


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files: 100%|██████████| 2/2 [00:25<00:00, 12.73s/it]


Loading weights:   1%|          | 3/434 [00:00<00:31, 13.78it/s, Materializing param=model.layers.0.mlp.down_proj.weight]

...

n_wait=1:  45%|████▌     | 9/20 [2:43:47<2:00:01, 654.70s/it]

n_wait=1:  50%|█████     | 10/20 [2:49:53<1:34:16, 565.62s/it]

n_wait=1:  55%|█████▌    | 11/20 [2:53:08<1:07:48, 452.03s/it]

n_wait=1:  60%|██████    | 12/20 [2:55:16<47:09, 353.69s/it]  

n_wait=1:  65%|██████▌   | 13/20 [3:05:42<50:52, 436.03s/it]

In [ ]:
# Cell 8 — Aggregate results
import glob, os

results_root = '/kaggle/working/s1_budget_forcing/experiments/results'
run_dirs = sorted(glob.glob(f'{results_root}/vi_*/'))

if not run_dirs:
    print('No vi_* result directories found. Run Cell 7 first.')
else:
    latest = run_dirs[-1]
    print(f'Aggregating: {latest}')
    !python experiments/results/summary_vi.py --results_dir {latest}

    import pandas as pd
    csv_path = os.path.join(latest, 'summary_vi.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        display_cols = ['model', 'benchmark', 'n_wait', 'n_samples',
                        'accuracy', 'scaling', 'performance',
                        'avg_thinking_tokens', 'extraction_failures']
        print(df[display_cols].to_string(index=False))
    else:
        print(f'CSV not found at {csv_path}')

In [ ]:
# Cell 9 — Copy outputs to /kaggle/working for download
# Kaggle allows downloading files from /kaggle/working/.
import shutil, os, glob

OUT = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

for d in glob.glob('/kaggle/working/s1_budget_forcing/experiments/results/vi_*'):
    dest = os.path.join(OUT, os.path.basename(d))
    if not os.path.exists(dest):
        shutil.copytree(d, dest)
        print(f'Copied: {dest}')

print(f'\nAll outputs in: {OUT}')
!ls -lh {OUT}